# A control loop you can turn the knobs on

An Arduino runs a 1 kHz control loop on a motor with a magnetic angle sensor.
This notebook sets the gains, runs an experiment, and gets the timeseries back.

Every cell starts with `sync_board()`, which rebuilds the sketch if you have
edited it, reflashes if the binary changed, and reopens the link — which resets
the board. So every cell starts from the sketch's defaults, and you can run them
in any order without wondering what the cell above left behind.

**The motor will move.** Check the bench before running anything below.

The controller itself is in `../ControlDemo/ControlDemo.ino`. That is the file to
edit when you want to change the control *law*; everything here changes only its
parameters.

In [ ]:
import sys
sys.path.insert(0, '../python')

import numpy as np
import matplotlib.pyplot as plt

from bench import *          # sync_board, and this rig's units

plt.rcParams['figure.figsize'] = (9, 3.5)
plt.rcParams['axes.grid'] = True

## 1. Does the hardware work?

Run this before anything else, and again after touching the wiring. It checks
each part of the rig separately — the loop's timing, the magnet, the I2C bus,
the current sense, and finally the motor — so a failure points at one thing
rather than at "the experiment didn't work".

It spins the motor briefly at the end. Pass `motor=False` to skip that.

In [ ]:
dev = sync_board()
dev.bringup()

## 2. The sensor

`y_uw` is the shaft angle, unwrapped: it counts through the 4096-count wrap
instead of jumping back to zero, so a shaft that keeps turning gives a line that
keeps rising. Turn the magnet by hand while this runs.

`y_uwf` is the same signal through a two-pole filter. `dev.smooth('y', tau)` sets
its time constant; `tau = 0` switches it off, which is the default — 50 ms of lag
is a lot of phase to give away on a 1 kHz loop, and you can always filter on this
side afterwards.

In [ ]:
dev = sync_board()
dev.zero()
dev.smooth('y', 0.05)           # 50 ms

df = dev.capture(3.0)

plt.plot(df['t'], df['y_uw'],  lw=0.8, label='y_uw   raw')
plt.plot(df['t'], df['y_uwf'],          label='y_uwf  filtered')
plt.xlabel('t [s]'); plt.ylabel('angle [deg]'); plt.legend()
plt.title('turn the magnet')
plt.show()

print(f'moved {df["y_uw"].max() - df["y_uw"].min():.1f} deg, '
      f'noise {df["y_uw"].diff().std():.3f} deg between samples')

## 3. Open loop: what does the plant do?

`mode = MODE_OPEN` disconnects the controller and puts `uff` straight on the
motor. Step it and watch the answer. That response — how fast it speeds up, how
much current it pulls — is what a controller has to be designed against, so this
is the measurement to take first.

`dev.step()` holds for `pre` seconds, changes the parameter, then holds for
`post`. The board reports the exact tick the change landed on, so `t = 0` is the
step itself to the sample; this side's timing jitter never enters the data.

In [ ]:
dev = sync_board()
dev.mode = MODE_OPEN

df = dev.step('uff', 200, pre=0.3, post=0.7, back=0)
dev.rest()

speed = np.diff(df['y_uw']) / (df.attrs['dt_us'] * 1e-6) / 360

fig, (a, b, c) = plt.subplots(3, 1, sharex=True, figsize=(9, 6))
a.plot(df['t'][1:], speed);   a.set_ylabel('speed [rev/s]')
b.plot(df['t'], df['u']);     b.set_ylabel('u [pwm]')
c.plot(df['t'], df['i']);     c.set_ylabel('i [mA]'); c.set_xlabel('t [s]')
for ax in (a, b, c):
    ax.axvline(0, color='k', lw=0.8, ls='--')
a.set_title('open-loop step: u = 0 -> 200')
plt.show()

## 4. Closing the loop on position

`mode = MODE_PID` runs the controller, and `target = POSITION` makes it work on
the angle. The error is `ref - y_uw`, both in counts; `dev.deg()` writes a
setpoint in degrees instead.

`dev.gains(kp, ki, kd)` takes the gains in continuous time — `ki` per second,
`kd` in seconds — and converts. The board's own `dev.kp`, `dev.ki` and `dev.kd`
are per *sample*, which is what its arithmetic actually multiplies by.

Start with kp alone. Add ki only once you can see a steady-state error worth
removing, and kd only if you can see oscillation worth damping.

In [ ]:
dev = sync_board()

dev.target = POSITION
dev.gains(kp=0.002, ki=0.0, kd=0.0)

dev.zero()
dev.ref  = 0
dev.mode = MODE_PID

df = dev.step('ref', dev.deg(90), pre=0.2, post=0.8, back=0)
dev.rest()

fig, (a, b) = plt.subplots(2, 1, sharex=True, figsize=(9, 5))
a.plot(df['t'], dev.as_deg(df['ref']), 'k--', lw=0.8, label='ref')
a.plot(df['t'], df['y_uw'], label='y_uw')
a.set_ylabel('angle [deg]'); a.legend()
b.plot(df['t'], df['u']); b.set_ylabel('u [pwm]'); b.set_xlabel('t [s]')
for ax in (a, b):
    ax.axvline(0, color='k', lw=0.8, ls='--')
a.set_title('closed-loop step to 90 deg')
plt.show()

settled = df['y_uw'].iloc[-1]
print(f'ended at {settled:.1f} deg, {90 - settled:+.1f} deg of steady-state error')

## 5. Sweeping a gain

The reason for driving this from a notebook: change one number, re-measure,
overlay. Each pass is one round trip to the board.

Watch what rising kp buys and what it costs — speed against overshoot, and
eventually against oscillation.

In [ ]:
dev = sync_board()

dev.target = POSITION
dev.zero()
dev.ref  = 0
dev.mode = MODE_PID

runs = {}
for kp in (0.0005, 0.001, 0.002, 0.004):
    dev.gains(kp=kp)
    runs[kp] = dev.step('ref', dev.deg(90), pre=0.1, post=0.6, back=0)

dev.rest()

for kp, d in runs.items():
    plt.plot(d['t'], d['y_uw'], label=f'kp = {kp}')
plt.axhline(90, color='k', lw=0.8, ls='--')
plt.axvline(0, color='k', lw=0.8, ls='--')
plt.xlabel('t [s]'); plt.ylabel('y_uw [deg]'); plt.legend(); plt.title('kp sweep')
plt.show()

for kp, d in runs.items():
    after = d[d['t'] > 0]['y_uw']
    print(f'kp={kp:<7} peak {after.max():6.1f} deg, final {after.iloc[-1]:6.1f} deg')

## 6. Following a ramp

`mode = MODE_RAMP` adds `refrate` to `ref` every control period, so the setpoint
sweeps at a constant speed and the loop has to *track* rather than settle.

A proportional controller cannot track a ramp without falling behind: the error
is what generates the command, so a constant command needs a constant error.
Adding ki is what closes that gap. Try it with `ki=0` first and watch the lag.

In [ ]:
dev = sync_board()

dev.target = POSITION
dev.gains(kp=0.002, ki=0.05)

dev.zero()
dev.ref     = 0
dev.refrate = dev.rev_per_s(1.0)
dev.mode    = MODE_RAMP

df = dev.step('refrate', dev.rev_per_s(2.0), pre=2.0, post=2.0)
dev.rest()

per_s = 1 / (df.attrs['dt_us'] * 1e-6) / 360

fig, (a, b) = plt.subplots(2, 1, sharex=True, figsize=(9, 5))
a.plot(df['t'][1:], np.diff(dev.as_deg(df['ref'])) * per_s, 'k--', lw=0.8, label='ref')
a.plot(df['t'][1:], np.diff(df['y_uw']) * per_s, label='y_uw')
a.set_ylabel('speed [rev/s]'); a.legend()
b.plot(df['t'], dev.as_deg(df['e'])); b.set_ylabel('e [deg]'); b.set_xlabel('t [s]')
for ax in (a, b):
    ax.axvline(0, color='k', lw=0.8, ls='--')
a.set_title('ramp: 1 rev/s, then 2')
plt.show()

## 7. The same controller, a different signal

`target = CURRENT` swaps the feedback: the same PID now works on `ref - i`, the
current through the motor. Nothing else changes — in the sketch it is one branch
in `target_error()` — so this is the same controller against a far faster plant,
and it needs different gains for exactly that reason.

`dev.ma()` writes the setpoint in milliamps. Current is noisy, so `alpha_i`
filters the measurement and `alpha_e` filters what the derivative term sees;
`dev.smooth()` sets either by time constant.

In [ ]:
dev = sync_board()

dev.target = CURRENT
dev.smooth('i', 0.005)
dev.smooth('e', 0.010)
dev.gains(kp=0.05, ki=2.0)

dev.ref  = dev.ma(150)
dev.mode = MODE_PID

df = dev.step('ref', dev.ma(300), pre=0.5, post=0.5, back=dev.ma(150))
dev.rest()

fig, (a, b) = plt.subplots(2, 1, sharex=True, figsize=(9, 5))
a.plot(df['t'], dev.as_ma(df['ref']), 'k--', lw=0.8, label='ref')
a.plot(df['t'], df['i'], label='i')
a.set_ylabel('current [mA]'); a.legend()
b.plot(df['t'], df['u']); b.set_ylabel('u [pwm]'); b.set_xlabel('t [s]')
for ax in (a, b):
    ax.axvline(0, color='k', lw=0.8, ls='--')
a.set_title('current step, 150 -> 300 mA')
plt.show()

## 8. What the sample rate does

`tickdiv` divides the 5 kHz sampler down to the control rate: `tickdiv = 5` is
1 kHz, `tickdiv = 50` is 100 Hz. The sketch's gains are *per sample*, so the same
three numbers mean something different at each rate — which is the point. A loop
tuned at 1 kHz and then run at 100 Hz is a loop with a tenth of the integral
action and ten times the derivative gain.

`dev.gains()` re-converts from continuous time, so calling it again after
changing `tickdiv` puts the loop back where it was. Comment it out to see what
happens if you forget.

In [ ]:
dev = sync_board()

dev.target = POSITION

for tickdiv in (5, 25, 50):
    dev.tickdiv = tickdiv
    dev.gains(kp=0.002, ki=0.05)      # re-converted for the new period
    dev.zero()
    dev.ref  = 0
    dev.mode = MODE_PID

    d = dev.step('ref', dev.deg(90), pre=0.1, post=0.6, back=0)
    plt.plot(d['t'], d['y_uw'], label=f'{1e6 / d.attrs["dt_us"]:.0f} Hz')

dev.rest()

plt.axhline(90, color='k', lw=0.8, ls='--')
plt.axvline(0, color='k', lw=0.8, ls='--')
plt.xlabel('t [s]'); plt.ylabel('y_uw [deg]'); plt.legend()
plt.title('the same gains at three loop rates')
plt.show()

## 9. Finishing up

Leave the motor at rest and free the port, or the next `sync_board()` will find
it busy.

In [ ]:
dev = sync_board()
dev.rest()
dev.close()
print('closed')